# 4. RAG Pipeline with ChromaDB
**Industry:** Healthcare

A RAG system that answers clinical questions from a custom protocol document.

In [ ]:
!pip install langchain langchain-google-genai chromadb sentence-transformers langchain-community

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# Create a mock clinical protocol document
with open('protocol.txt', 'w') as f:
    f.write("""Clinical Protocol for Hypertension:\n1. First-line treatment for uncomplicated hypertension is an ACE inhibitor or ARB.\n2. If patient is over 55 or of African family origin, use Calcium Channel Blocker (CCB) first.\n3. Target blood pressure is <140/90 mmHg for most patients.""")

loader = TextLoader("protocol.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the first-line treatment for a 60-year-old patient?"
print(rag_chain.invoke(question))